# exp007: A vs. B nas 15 categorias do MVTec AD (Colab)

Experimento intermediário antes do Artigo 1. 740 imagens (50 por categoria, 40 no `toothbrush`), 3 perguntas, Configurações A e B.

Use GPU (Ambiente de execução > Alterar o tipo). L4 ou A100 rodam em bf16; T4 cai para fp16. Tudo que é resultado vai para o Drive, então dá para retomar se a sessão cair: é só rodar de novo do início.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ic-visionexplain'
!mkdir -p $DRIVE/exp007_multicategoria $DRIVE/anomalib_runs

In [ ]:
%cd /content
!test -d vision-explain || git clone https://github.com/Edryck/vision-explain.git
%cd /content/vision-explain
!git pull --ff-only
!pip install -q -r requirements.txt

## Dados

Suba `mvtec-ad.zip` para `MyDrive/ic-visionexplain/`. Se não houver zip, o anomalib baixa o MVTec sozinho no primeiro treino (mais lento).

In [ ]:
import os
!mkdir -p data/mvtec_ad
if not os.path.isdir('data/mvtec_ad/mvtec-ad/bottle') and os.path.exists(f'{DRIVE}/mvtec-ad.zip'):
    !cp $DRIVE/mvtec-ad.zip /content/mvtec-ad.zip
    !unzip -q /content/mvtec-ad.zip -d data/mvtec_ad/mvtec-ad/
!ls data/mvtec_ad/mvtec-ad | head -20

Resultados e checkpoints do PatchCore ficam no Drive via link simbólico. O `subset.json` versionado no repo é copiado só se o Drive ainda não tiver um.

In [ ]:
!test -f $DRIVE/exp007_multicategoria/subset.json || cp experiments/exp007_multicategoria/subset.json $DRIVE/exp007_multicategoria/
!rm -rf experiments/exp007_multicategoria data/anomalib_runs
!ln -s $DRIVE/exp007_multicategoria experiments/exp007_multicategoria
!ln -s $DRIVE/anomalib_runs data/anomalib_runs
!ls experiments/exp007_multicategoria

In [ ]:
import torch
print(torch.cuda.get_device_name(0), 'bf16' if torch.cuda.is_bf16_supported() else 'fp16')

## 1. Evidências do PatchCore

Treina um PatchCore por categoria (coreset 0,01, igual ao bottle), roda o teste completo (AUROC de imagem e de pixel) e extrai a evidência das imagens do subconjunto.

In [ ]:
!python -m src.configB.exp007_multicategoria.evidences

## 2. Configuração A (imagem + pergunta)

In [ ]:
!python -m src.configA.exp007_multicategoria.run

## 3. Configuração B (imagem + evidência + pergunta)

In [ ]:
!python -m src.configB.exp007_multicategoria.run

## 4. Métricas

Acerto por gabarito, FaithScore simplificado, fidelidade à evidência, consistência entre perguntas e riqueza. Depois, McNemar e bootstrap por imagem.

In [ ]:
!python -m src.multicategory_metrics score --config A --faithscore
!python -m src.multicategory_metrics score --config B --faithscore
!python -m src.multicategory_metrics report

In [ ]:
import pandas as pd
pd.read_csv('experiments/exp007_multicategoria/results/tabela_por_categoria.csv')

In [ ]:
import json
summary = json.load(open('experiments/exp007_multicategoria/results/resumo_exp007.json'))
print(json.dumps({'overall': summary['overall'], 'paired': summary['paired']}, indent=1)[:6000])

Ao final, baixe a pasta `MyDrive/ic-visionexplain/exp007_multicategoria/` para `experiments/exp007_multicategoria/` no repo local. Os checkpoints em `anomalib_runs` não vão para o git.